# CUPED for Experimentation
- Variance Reduction using CUPED

## Create Testing Dataset

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 5000  # users per group

def generate_users(n, treatment=False):
    # Base spending tendency per user (log-normal = realistic whale distribution)
    base_spend = np.random.lognormal(mean=1.5, sigma=1.0, size=n)

    # Pre-experiment spend (30 days before): correlated with base + noise
    pre_spend = np.maximum(0, base_spend + np.random.normal(0, 8, n))

    # In-experiment spend: driven by base + noise + optional treatment effect
    treatment_effect = 1.80 if treatment else 0.0
    in_spend = np.maximum(0, base_spend * 0.75 + np.random.normal(0, 6, n) + treatment_effect)

    return pd.DataFrame({
        "user_id": [f"{'T' if treatment else 'C'}_{i:04d}" for i in range(n)],
        "group": "treatment" if treatment else "control",
        "pre_spend": pre_spend,
        "in_spend": in_spend,
    })

control   = generate_users(N, treatment=False)
treatment = generate_users(N, treatment=True)
df = pd.concat([control, treatment], ignore_index=True)

# df.to_csv("experiment_data.csv", index=False)
# print(df.groupby("group")[["pre_spend", "in_spend"]].describe().round(2))
# print(f"\nDataset saved: {len(df):,} users")

          pre_spend                                            in_spend        \
              count  mean    std  min  25%   50%    75%    max    count  mean   
group                                                                           
control      5000.0  8.75  11.31  0.0  0.0  5.89  13.00  222.9   5000.0  6.62   
treatment    5000.0  8.73  12.10  0.0  0.0  6.02  12.98  405.9   5000.0  8.08   

                                                 
            std  min   25%   50%    75%     max  
group                                            
control    8.61  0.0  0.00  4.56   9.60  176.43  
treatment  9.72  0.0  1.31  6.26  11.75  307.12  


## Run a t-test

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv("experiment_data.csv")
ctrl = df[df.group == "control"]["in_spend"]
trt  = df[df.group == "treatment"]["in_spend"]

t_stat, p_val = stats.ttest_ind(ctrl, trt)
diff = trt.mean() - ctrl.mean()
pooled_std = df["in_spend"].std()

print("=" * 45)
print("         STANDARD T-TEST RESULTS")
print("=" * 45)
print(f"  Control mean:    ${ctrl.mean():.4f}")
print(f"  Treatment mean:  ${trt.mean():.4f}")
print(f"  Observed lift:   ${diff:.4f}")
print(f"  Variance:        {df['in_spend'].var():.2f}")
print(f"  t-statistic:     {t_stat:.4f}")
print(f"  p-value:         {p_val:.4f}")
print(f"  Significant:     {'YES ✓' if p_val < 0.05 else 'NO ✗'} (α = 0.05)")
print("=" * 45)